# 2D Mobile Robot Kalman Filter 框架

这个 notebook 先只放文字框架，后续逐步补代码。

目标：从 1D 小车扩展到 2D 移动机器人位置估计。

核心状态：

`x = [px, py, vx, vy]`

核心测量：

`z = [measured_px, measured_py]`

最终输出：

- `figures/kf_2d_trajectory.png`
- `figures/kf_2d_error.png`
- `results/kf_2d_rmse.csv`

## 1. 导入依赖

这一节后续需要导入：

- `Path`：管理项目路径、图片路径、结果路径
- `numpy`：生成轨迹、矩阵、噪声、误差
- `pandas`：保存 RMSE 表格
- `matplotlib.pyplot`：画轨迹图和误差图
- `KalmanFilter`：FilterPy 的核心滤波器类

参考来源：

- 1D notebook 的导入结构
- `source/filterpy/filterpy/kalman/tests/test_kf.py` 里 `KalmanFilter` 的使用方式

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"

FIGURES_DIR.mkdir(parents = True, exist_ok = True)
RESULTS_DIR.mkdir(parents = True, exist_ok = True)

PROJECT_ROOT

WindowsPath('e:/申博材料/机器人相关学习/阶段一_基础补齐/01_FilterPy状态估计_移动机器人定位')

## 2. 设置实验参数

这一节先定义实验参数，不做计算。

建议参数：

- `dt`：时间间隔
- `num_steps`：仿真步数
- `initial_position`：初始二维位置 `[px0, py0]`
- `true_velocity`：真实二维速度 `[vx, vy]`
- `initial_velocity_guess`：滤波器一开始猜的速度
- `measurement_noise_std`：位置测量噪声标准差
- `process_noise_var`：过程噪声方差
- `random_seed`：随机种子，保证实验可复现

核心理解：

`true_velocity` 是真实世界里的速度；`initial_velocity_guess` 是滤波器一开始的猜测，两者可以不一样。

In [4]:
dt = 1.0
num_steps = 80
initial_position = np.array([0.0, 0.0])
true_velocity = np.array([1.0, 1.0])
initial_velocity_guess = np.array([0.0, 0.0])

measurement_noise_std = 4.0
process_noise_var = 0.01
random_seed = 42

rng = np.random.default_rng(random_seed)

## 3. generate_2d_ground_truth

函数目标：生成二维真实轨迹。

建议函数名：

`generate_2d_ground_truth()`

输入：

- `num_steps`
- `dt`
- `initial_position`
- `true_velocity`

输出：

- `times`：时间序列，形状 `(num_steps,)`
- `true_positions`：二维真实位置，形状 `(num_steps, 2)`

流程逻辑：

1. 生成时间序列
2. 根据二维匀速公式生成真实位置
3. 返回 `times` 和 `true_positions`

要用到的函数：

- `np.arange()`：生成时间下标
- NumPy 广播机制：把时间序列和二维速度组合成二维轨迹

不需要用到：

- `KalmanFilter`
- `predict()`
- `update()`

对应模型：

`px = px0 + vx * t`

`py = py0 + vy * t`

In [5]:
def generate_2d_ground_truth(num_steps, dt, initial_position, true_velocity):
    times = np.arange(num_steps) * dt
    true_positions = initial_position + times[:, None] * true_velocity
    return times, true_positions


times, true_positions = generate_2d_ground_truth(
    num_steps=num_steps,
    dt=dt,
    initial_position=initial_position,
    true_velocity=true_velocity,
)

times[:5], true_positions[:5]

(array([0., 1., 2., 3., 4.]),
 array([[0., 0.],
        [1., 1.],
        [2., 2.],
        [3., 3.],
        [4., 4.]]))

## 4. add_position_measurement_noise

函数目标：在真实二维位置上叠加高斯噪声，模拟传感器测量。

建议函数名：

`add_position_measurement_noise()`

输入：

- `true_positions`
- `measurement_noise_std`
- `rng`

输出：

- `measurements`：带噪声二维位置测量，形状 `(num_steps, 2)`

流程逻辑：

1. 生成和 `true_positions` 形状相同的高斯噪声
2. 把噪声加到真实位置上
3. 返回测量值

要用到的函数：

- `rng.normal()`：生成高斯噪声

对应 Kalman Filter 里的概念：

- 这里生成的是后续 `update(z)` 里的 `z`
- 这里的 `measurement_noise_std` 后续要和 `R` 对应
- `R` 用的是方差，即 `measurement_noise_std ** 2`

In [8]:
def add_position_measurement_noise(true_positions, measurement_noise_std, rng):
    noise = rng.normal(
        loc = 0.0,
        scale = measurement_noise_std,
        size = true_positions.shape
    )
    measurements = true_positions + noise
    return measurements

measurements = add_position_measurement_noise(
    true_positions = true_positions,
    measurement_noise_std = measurement_noise_std,
    rng = rng,
)

measurements[:5]

array([[ 3.12940137, -0.76260423],
       [ 5.68498837,  4.00347595],
       [ 9.28258463,  4.92309876],
       [-3.28816102,  2.73218731],
       [-0.68802877,  1.92688063]])

## 5. create_2d_kalman_filter

函数目标：创建并配置二维匀速模型的 Kalman Filter。

建议函数名：

`create_2d_kalman_filter()`

输入：

- `dt`
- `initial_position`
- `initial_velocity`
- `measurement_noise_std`
- `process_noise_var`

输出：

- 配置好的 `kf`

需要配置的内容：

1. 创建 `KalmanFilter(dim_x=4, dim_z=2)`
2. 设置初始状态 `kf.x`
3. 设置状态转移矩阵 `kf.F`
4. 设置观测矩阵 `kf.H`
5. 设置初始不确定性 `kf.P`
6. 设置测量噪声 `kf.R`
7. 设置过程噪声 `kf.Q`

状态定义：

`x = [px, py, vx, vy]`

测量定义：

`z = [px, py]`

状态转移矩阵：

```text
F = [[1, 0, dt, 0 ],
     [0, 1, 0,  dt],
     [0, 0, 1,  0 ],
     [0, 0, 0,  1 ]]
```

观测矩阵：

```text
H = [[1, 0, 0, 0],
     [0, 1, 0, 0]]
```

要用到的现有函数：

- `KalmanFilter`：来自 `filterpy.kalman`
- `np.array()`：定义状态和矩阵
- `np.eye()`：定义 `P` 和 `R`

关于 `Q`：

2D 中如果状态顺序是 `[px, py, vx, vy]`，`Q` 要注意矩阵维度和状态顺序对应。可以先用简单版 `np.eye(4) * process_noise_var`，跑通后再换成更严格的 constant-velocity 过程噪声矩阵。

In [11]:
def create_2d_kalman_filter(dt, initial_position, initial_velocity, 
                            measurement_noise_std, process_noise_var):
    kf = KalmanFilter(dim_x = 4, dim_z = 2)

    kf.x = np.array([
        initial_position[0],
        initial_position[1],
        initial_velocity[0],
        initial_velocity[1],
    ])
    kf.F = np.array([
        [1.0, 0.0, dt, 0.0],
        [0.0, 1.0, 0.0, dt],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ])
    kf.H = np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
    ])
    kf.P = np.eye(4) * 500.0
    kf.R = np.eye(2) * measurement_noise_std ** 2
    kf.Q = np.eye(4) * process_noise_var

    return kf

kf = create_2d_kalman_filter(
    dt = dt,
    initial_position = initial_position,
    initial_velocity = initial_velocity_guess,
    measurement_noise_std = measurement_noise_std,
    process_noise_var = process_noise_var
)

print("x shape:", kf.x.shape)
print("F shape:", kf.F.shape)
print("H shape:", kf.H.shape)
print("P shape:", kf.P.shape)
print("R shape:", kf.R.shape)
print("Q shape:", kf.Q.shape)

x shape: (4,)
F shape: (4, 4)
H shape: (2, 4)
P shape: (4, 4)
R shape: (2, 2)
Q shape: (4, 4)


## 6. run_2d_kalman_filter

函数目标：对所有二维测量值执行 Kalman Filter。

建议函数名：

`run_2d_kalman_filter()`

输入：

- `measurements`
- `dt`
- `initial_position`
- `initial_velocity`
- `measurement_noise_std`
- `process_noise_var`

输出：

- `estimates`：估计状态，形状 `(num_steps, 4)`

流程逻辑：

1. 调用 `create_2d_kalman_filter()` 创建滤波器
2. 创建数组保存每一步状态估计
3. 遍历每一个二维测量值 `z`
4. 每一步先执行 `kf.predict()`
5. 再执行 `kf.update(z)`
6. 保存当前 `kf.x`
7. 返回 `estimates`

要用到的函数：

- 自己写的 `create_2d_kalman_filter()`
- `kf.predict()`：FilterPy 方法
- `kf.update(z)`：FilterPy 方法
- `np.zeros()`：保存估计结果
- `enumerate()`：同时拿到下标和测量值

注意：

二维测量 `z` 应该是 `[measured_px, measured_py]`，形状是 `(2,)` 或 `(2, 1)`，和 `dim_z=2` 对应。

In [12]:
def run_2d_kalman_filter(measurements, dt, initial_position, initial_velocity,
                         measurement_noise_std, process_noise_var):
    kf = create_2d_kalman_filter(
        dt = dt,
        initial_position = initial_position,
        initial_velocity = initial_velocity,
        measurement_noise_std = measurement_noise_std,
        process_noise_var = process_noise_var,
    )

    estimates = np.zeros((len(measurements), 4))

    for i, z in enumerate(measurements):
        kf.predict()
        kf.update(z)

        estimates[i] = kf.x

    return estimates

estimates = run_2d_kalman_filter(
    measurements=measurements,
    dt=dt,
    initial_position=initial_position,
    initial_velocity=initial_velocity_guess,
    measurement_noise_std=measurement_noise_std,
    process_noise_var=process_noise_var,
)

estimates[:5]

array([[ 3.08011994, -0.75059483,  1.54004457, -0.37529366],
       [ 5.62847194,  3.73123055,  2.46487731,  4.07971939],
       [ 9.06006746,  5.46344299,  3.02748646,  2.71352097],
       [ 1.48094637,  4.42100144, -1.44318595,  1.13038725],
       [-0.39351616,  3.39764283, -1.5854822 ,  0.41977607]])

## 7. compute_position_rmse

函数目标：计算二维位置 RMSE。

建议函数名：

`compute_position_rmse()`

输入：

- `estimated_positions`
- `true_positions`

输出：

- 一个标量 `rmse`

流程逻辑：

1. 计算每一步二维位置误差
2. 把 `x/y` 方向误差合成欧氏距离误差
3. 对所有时间步求均方根

要用到的函数：

- `np.sum()`
- `np.mean()`
- `np.sqrt()`

需要比较两组 RMSE：

- `measurement_rmse`：测量值相对真实轨迹的 RMSE
- `kf_rmse`：滤波估计相对真实轨迹的 RMSE

期望：

`kf_rmse < measurement_rmse`

In [14]:
def compute_position_rmse(estimated_positions, true_positions):
    position_errors = estimated_positions - true_positions
    squared_distances = np.sum(position_errors ** 2, axis=1)
    mean_squared_distance = np.mean(squared_distances)
    rmse = np.sqrt(mean_squared_distance)
    return rmse

measurement_rmse = compute_position_rmse(
    estimated_positions=measurements,
    true_positions=true_positions,
)

kf_rmse = compute_position_rmse(
    estimated_positions=estimates[:, :2],
    true_positions=true_positions,
)

## 8. 保存 RMSE 表格

目标输出：

`results/kf_2d_rmse.csv`

建议表格结构：

```csv
method,position_rmse
noisy_measurement,...
kalman_filter,...
```

要用到的函数：

- `pd.DataFrame()`
- `DataFrame.to_csv()`
- `Path.mkdir()`：确保 `results/` 目录存在

## 9. plot_trajectory

函数目标：画二维轨迹图。

建议函数名：

`plot_trajectory()`

输入：

- `true_positions`
- `measurements`
- `estimates`
- `output_path`

图中至少包含：

- `Ground truth`
- `Noisy measurement`
- `Kalman estimate`

目标输出：

`figures/kf_2d_trajectory.png`

要用到的函数：

- `plt.figure()`
- `plt.plot()`
- `plt.scatter()`
- `plt.axis("equal")`
- `plt.legend()`
- `plt.grid()`
- `plt.savefig()`
- `plt.show()`

关键点：

- `true_positions[:, 0]` 是真实 x
- `true_positions[:, 1]` 是真实 y
- `estimates[:, 0]` 是估计 x
- `estimates[:, 1]` 是估计 y

## 10. plot_position_error

函数目标：画误差随时间变化的曲线。

建议函数名：

`plot_position_error()`

输入：

- `times`
- `true_positions`
- `measurements`
- `estimates`
- `output_path`

图中建议包含：

- measurement error
- Kalman error

目标输出：

`figures/kf_2d_error.png`

要用到的函数：

- `np.linalg.norm()`：计算二维误差长度
- `plt.plot()`
- `plt.savefig()`

核心理解：

轨迹图是直观展示；误差图和 RMSE 是量化证明。

## 11. 后续转成 Python 脚本的结构

等 notebook 跑通后，再整理成：

`scripts/02_kf_2d_mobile_robot.py`

建议脚本函数结构：

```text
generate_2d_ground_truth()
add_position_measurement_noise()
create_2d_kalman_filter()
run_2d_kalman_filter()
compute_position_rmse()
plot_trajectory()
plot_position_error()
main()
```

优先级：

1. 先让 notebook 每一步能跑通
2. 再确认图像和 RMSE 合理
3. 最后再整理成完整 `.py` 文件